# Этап 2: Сравнение моделей для поиска семантических проблем

Этот ноутбук нужен как промежуточный шаг: выбрать модель эмбеддингов для основной цели проекта.
Добавлено: multi-seed устойчивость и сохранение итогов в out/metrics_summary.json.
\n

In [1]:
import json
from pathlib import Path

import numpy as np
from compute_embed import load_meta, load_embeddings


TAGS = [
    "rubert_base",
    "rubert_tiny2",
    "sbert_large_nlu_ru",
    "FRIDA",
    "all_mpnet_v2",
    "multi_mpnet_v2",
]

BASE = Path("out")
META_PATH = BASE / "meta.json"
READY = BASE / "metrics_ready"
READY.mkdir(parents=True, exist_ok=True)

RANDOM_SEEDS = list(range(101, 131))

print("models:", TAGS)
print("random seeds:", len(RANDOM_SEEDS), f"({RANDOM_SEEDS[0]}..{RANDOM_SEEDS[-1]})")


g:\Dipl\parser\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


models: ['rubert_base', 'rubert_tiny2', 'sbert_large_nlu_ru', 'FRIDA', 'all_mpnet_v2', 'multi_mpnet_v2']
random seeds: 30 (101..130)


## 1) Построение S и adj

Здесь строятся базовые артефакты:
- S (sim_<tag>.npy): матрица cosine similarity между сегментами;
- adj (adj_<tag>.npy): близость соседних сегментов S[i, i+1].
\n

In [2]:
meta = load_meta(str(META_PATH))


def cosine_matrix(emb: np.ndarray) -> np.ndarray:
    emb = emb.astype(np.float32)
    emb = emb / (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-9)
    return emb @ emb.T


for tag in TAGS:
    emb_path = BASE / f"emb_{tag}.npy"
    if not emb_path.exists():
        print(tag, "skip: missing", emb_path)
        continue

    emb = load_embeddings(str(emb_path))
    S = cosine_matrix(emb)
    adj = np.array([float(S[i, i + 1]) for i in range(len(meta) - 1)], dtype=np.float32)

    np.save(READY / f"sim_{tag}.npy", S)
    np.save(READY / f"adj_{tag}.npy", adj)

    print(tag, "ok", S.shape, "adj_mean", round(float(adj.mean()) if adj.size else float('nan'), 4))


rubert_base ok (11, 11) adj_mean 0.9268
rubert_tiny2 ok (11, 11) adj_mean 0.8661
sbert_large_nlu_ru ok (11, 11) adj_mean 0.8619
FRIDA ok (11, 11) adj_mean 0.7207
all_mpnet_v2 ok (11, 11) adj_mean 0.6424
multi_mpnet_v2 ok (11, 11) adj_mean 0.6621


## 2) Вспомогательные функции

Ячейка содержит функции для метрик и выборок:
- эффект Коэна d и KS-статистика;
- выборка случайных несоседних пар;
- извлечение ролей intro/main/conclusion.
\n

In [3]:
def ks_statistic(a: np.ndarray, b: np.ndarray) -> float:
    a = np.sort(a.astype(np.float64))
    b = np.sort(b.astype(np.float64))
    n = a.size
    m = b.size
    i = 0
    j = 0
    d = 0.0

    while i < n and j < m:
        if a[i] <= b[j]:
            i += 1
        else:
            j += 1
        d = max(d, abs(i / n - j / m))

    return float(d)


def cohen_d(a: np.ndarray, b: np.ndarray) -> float:
    a = a.astype(np.float64)
    b = b.astype(np.float64)
    va = a.var(ddof=1) if a.size > 1 else 0.0
    vb = b.var(ddof=1) if b.size > 1 else 0.0
    sp = np.sqrt((va + vb) / 2.0) + 1e-12
    return float((a.mean() - b.mean()) / sp)


def sample_random_pairs(S: np.ndarray, k: int, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    n = S.shape[0]
    vals = []
    tries = 0
    need = int(k)

    while need > 0 and tries < max(100, k * 20):
        i = int(rng.integers(0, n))
        j = int(rng.integers(0, n))
        tries += 1

        if i == j:
            continue
        if abs(i - j) == 1:
            continue

        vals.append(float(S[i, j]))
        need -= 1

    return np.array(vals, dtype=np.float32)


def get_role_indices(meta_items):
    roles = np.array([m.get("role", "") for m in meta_items])
    main_idx = np.where(roles == "main")[0]
    intro_idx = np.where(roles == "intro")[0]
    conc_idx = np.where(roles == "conclusion")[0]

    intro_i = int(intro_idx[0]) if intro_idx.size else None
    conc_i = int(conc_idx[0]) if conc_idx.size else None

    return main_idx, intro_i, conc_i


main_idx, intro_i, conc_i = get_role_indices(meta)
print("segments:", len(meta), "main:", len(main_idx), "intro:", intro_i, "conclusion:", conc_i)


segments: 11 main: 9 intro: 0 conclusion: 10


## 3) Расчет метрик по всем моделям (multi-seed)

Для каждого seed считаются rand, d, ks, delta, intro_delta и conc_delta.
Потом результаты агрегируются в mean/std, чтобы сравнение было устойчивым.
\n

In [4]:
rows = []

for tag in TAGS:
    sim_path = READY / f"sim_{tag}.npy"
    adj_path = READY / f"adj_{tag}.npy"

    if not sim_path.exists() or not adj_path.exists():
        print(tag, "skip: missing sim/adj files")
        continue

    S = np.load(sim_path)
    adj = np.load(adj_path).astype(np.float32)

    if adj.size == 0:
        print(tag, "skip: adj is empty")
        continue

    d_vals = []
    ks_vals = []
    delta_vals = []
    delta_med_vals = []
    rand_mean_vals = []
    intro_delta_vals = []
    conc_delta_vals = []
    score_vals = []

    k_rand = max(2000, 5 * len(adj))

    for seed in RANDOM_SEEDS:
        rand = sample_random_pairs(S, k=k_rand, seed=seed)
        if rand.size == 0:
            continue

        d = cohen_d(adj, rand)
        ks = ks_statistic(adj, rand)
        delta_mean = float(adj.mean() - rand.mean())
        delta_med = float(np.median(adj) - np.median(rand))

        intro_delta = float("nan")
        conc_delta = float("nan")

        if intro_i is not None and main_idx.size:
            intro_to_main = S[intro_i, main_idx].astype(np.float32)
            base_main = sample_random_pairs(S[np.ix_(main_idx, main_idx)], k=2000, seed=seed + 10000)
            if base_main.size:
                intro_delta = float(intro_to_main.mean() - base_main.mean())

        if conc_i is not None and main_idx.size:
            conc_to_main = S[conc_i, main_idx].astype(np.float32)
            base_main = sample_random_pairs(S[np.ix_(main_idx, main_idx)], k=2000, seed=seed + 20000)
            if base_main.size:
                conc_delta = float(conc_to_main.mean() - base_main.mean())

        redund_90 = float("nan")
        if main_idx.size >= 2:
            M = S[np.ix_(main_idx, main_idx)]
            iu = np.triu_indices_from(M, k=1)
            main_sims = M[iu].astype(np.float32)
            redund_90 = float(np.mean(main_sims > 0.90))

        score = (
            0.45 * d
            + 0.25 * ks
            + 0.15 * delta_mean
            + 0.10 * (intro_delta if not np.isnan(intro_delta) else 0.0)
            + 0.10 * (conc_delta if not np.isnan(conc_delta) else 0.0)
            - 0.25 * (redund_90 if not np.isnan(redund_90) else 0.0)
        )

        d_vals.append(d)
        ks_vals.append(ks)
        delta_vals.append(delta_mean)
        delta_med_vals.append(delta_med)
        rand_mean_vals.append(float(rand.mean()))
        intro_delta_vals.append(intro_delta)
        conc_delta_vals.append(conc_delta)
        score_vals.append(score)

    if not score_vals:
        print(tag, "skip: no valid score values")
        continue

    def _mean_std(values):
        arr = np.array(values, dtype=np.float64)
        arr = arr[~np.isnan(arr)]
        if arr.size == 0:
            return float("nan"), float("nan")
        return float(arr.mean()), float(arr.std(ddof=0))

    d_mean, d_std = _mean_std(d_vals)
    ks_mean, ks_std = _mean_std(ks_vals)
    delta_mean, delta_std = _mean_std(delta_vals)
    delta_med_mean, delta_med_std = _mean_std(delta_med_vals)
    rand_mean, rand_std = _mean_std(rand_mean_vals)
    intro_mean, intro_std = _mean_std(intro_delta_vals)
    conc_mean, conc_std = _mean_std(conc_delta_vals)
    score_mean, score_std = _mean_std(score_vals)

    redund_90 = float("nan")
    if main_idx.size >= 2:
        M = S[np.ix_(main_idx, main_idx)]
        iu = np.triu_indices_from(M, k=1)
        main_sims = M[iu].astype(np.float32)
        redund_90 = float(np.mean(main_sims > 0.90))

    rows.append(
        {
            "tag": tag,
            "score_mean": score_mean,
            "score_std": score_std,
            "adj_mean": float(adj.mean()),
            "adj_std": float(adj.std(ddof=0)),
            "rand_mean": rand_mean,
            "rand_std": rand_std,
            "delta_mean": delta_mean,
            "delta_std": delta_std,
            "delta_med_mean": delta_med_mean,
            "delta_med_std": delta_med_std,
            "effect_d_mean": d_mean,
            "effect_d_std": d_std,
            "ks_mean": ks_mean,
            "ks_std": ks_std,
            "intro_delta_mean": intro_mean,
            "intro_delta_std": intro_std,
            "conc_delta_mean": conc_mean,
            "conc_delta_std": conc_std,
            "redund_90": redund_90,
            "n_seeds": len(score_vals),
        }
    )

print("models with computed metrics:", len(rows))


models with computed metrics: 6


## 4) Метрика effect_d

Показывает силу разделения соседних и случайных пар.
Чем выше effect_d_mean, тем лучше модель улавливает структурную связность.
\n

In [5]:
for r in sorted(rows, key=lambda x: x["effect_d_mean"], reverse=True):
    print(r["tag"], "effect_d", round(r["effect_d_mean"], 3), "?", round(r["effect_d_std"], 3))


FRIDA effect_d 0.713 ? 0.03
multi_mpnet_v2 effect_d 0.673 ? 0.025
rubert_tiny2 effect_d 0.531 ? 0.026
sbert_large_nlu_ru effect_d 0.514 ? 0.016
rubert_base effect_d 0.417 ? 0.021
all_mpnet_v2 effect_d 0.054 ? 0.024


## 5) Метрика ks

KS сравнивает распределения adj и rand целиком.
Чем выше ks_mean, тем сильнее различие между связными и случайными парами.
\n

In [6]:
for r in sorted(rows, key=lambda x: x["ks_mean"], reverse=True):
    print(r["tag"], "ks", round(r["ks_mean"], 3), "?", round(r["ks_std"], 3))


FRIDA ks 0.556 ? 0.01
sbert_large_nlu_ru ks 0.533 ? 0.009
rubert_tiny2 ks 0.513 ? 0.013
rubert_base ks 0.412 ? 0.014
multi_mpnet_v2 ks 0.41 ? 0.01
all_mpnet_v2 ks 0.301 ? 0.012


## 6) Метрика delta_mean

Это разница средних: adj_mean - rand_mean.
Чем выше delta_mean, тем сильнее локальная семантическая связность.
\n

In [7]:
for r in sorted(rows, key=lambda x: x["delta_mean"], reverse=True):
    print(
        r["tag"],
        "delta", round(r["delta_mean"], 3), "?", round(r["delta_std"], 3),
        "adj", round(r["adj_mean"], 3),
        "rand", round(r["rand_mean"], 3),
    )


FRIDA delta 0.083 ? 0.003 adj 0.721 rand 0.637
multi_mpnet_v2 delta 0.081 ? 0.003 adj 0.662 rand 0.581
sbert_large_nlu_ru delta 0.036 ? 0.001 adj 0.862 rand 0.826
rubert_tiny2 delta 0.033 ? 0.002 adj 0.866 rand 0.833
rubert_base delta 0.019 ? 0.001 adj 0.927 rand 0.908
all_mpnet_v2 delta 0.007 ? 0.003 adj 0.642 rand 0.636


## 7) Метрики intro_delta и conc_delta

Показывают согласованность введения и заключения с основной частью.
Более высокие значения означают лучшую структурную целостность.
\n

In [8]:
print("intro_delta ranking")
for r in sorted(rows, key=lambda x: x["intro_delta_mean"], reverse=True):
    print(r["tag"], "intro_delta", round(r["intro_delta_mean"], 3), "?", round(r["intro_delta_std"], 3))

print()
print("conc_delta ranking")
for r in sorted(rows, key=lambda x: x["conc_delta_mean"], reverse=True):
    print(r["tag"], "conc_delta", round(r["conc_delta_mean"], 3), "?", round(r["conc_delta_std"], 3))


intro_delta ranking
FRIDA intro_delta 0.121 ? 0.002
rubert_tiny2 intro_delta 0.059 ? 0.001
rubert_base intro_delta 0.034 ? 0.001
all_mpnet_v2 intro_delta 0.031 ? 0.002
multi_mpnet_v2 intro_delta 0.023 ? 0.003
sbert_large_nlu_ru intro_delta 0.007 ? 0.001

conc_delta ranking
FRIDA conc_delta 0.116 ? 0.002
multi_mpnet_v2 conc_delta 0.111 ? 0.003
all_mpnet_v2 conc_delta 0.046 ? 0.003
rubert_tiny2 conc_delta 0.035 ? 0.002
rubert_base conc_delta 0.022 ? 0.001
sbert_large_nlu_ru conc_delta -0.012 ? 0.001


## 8) Метрика redund_90

Доля слишком похожих пар внутри main (порог > 0.90).
Чем ниже значение, тем меньше риск семантического схлопывания.
\n

In [9]:
for r in sorted(rows, key=lambda x: x["redund_90"]):
    print(r["tag"], "redund_90", round(r["redund_90"], 3))


FRIDA redund_90 0.0
all_mpnet_v2 redund_90 0.0
multi_mpnet_v2 redund_90 0.0
rubert_tiny2 redund_90 0.083
sbert_large_nlu_ru redund_90 0.111
rubert_base redund_90 0.611


## 9) Итоговый score и сохранение отчета

Финальная ячейка строит рейтинг моделей по score_mean
и сохраняет полный отчет в out/metrics_summary.json.
\n

In [10]:
rows = sorted(rows, key=lambda x: x["score_mean"], reverse=True)

for r in rows:
    print(
        r["tag"],
        "score", round(r["score_mean"], 3), "?", round(r["score_std"], 3),
        "d", round(r["effect_d_mean"], 3),
        "ks", round(r["ks_mean"], 3),
        "delta", round(r["delta_mean"], 3),
        "redund_90", round(r["redund_90"], 3),
    )

if rows:
    print()
    print("best:", rows[0]["tag"])

summary = {
    "purpose": "model_selection_for_semantic_problem_detection",
    "random_seeds": RANDOM_SEEDS,
    "n_models": len(rows),
    "ranking": rows,
}

summary_path = BASE / "metrics_summary.json"
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print("saved:", summary_path)


FRIDA score 0.496 ? 0.016 d 0.713 ks 0.556 delta 0.083 redund_90 0.0
multi_mpnet_v2 score 0.431 ? 0.014 d 0.673 ks 0.41 delta 0.081 redund_90 0.0
rubert_tiny2 score 0.361 ? 0.015 d 0.531 ks 0.513 delta 0.033 redund_90 0.083
sbert_large_nlu_ru score 0.342 ? 0.009 d 0.514 ks 0.533 delta 0.036 redund_90 0.111
rubert_base score 0.146 ? 0.013 d 0.417 ks 0.412 delta 0.019 redund_90 0.611
all_mpnet_v2 score 0.108 ? 0.014 d 0.054 ks 0.301 delta 0.007 redund_90 0.0

best: FRIDA
saved: out\metrics_summary.json
